In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

# Establish connection
client = bigquery.Client()




Task 2: Connect Python to BigQuery and extract training data
Your Python environment needs access to the sales history you just created. You install the required libraries and establish the connection to pull data for model training.

HINT
Install three libraries: pip install google-cloud-bigquery pandas scikit-learn statsmodels. The statsmodels library contains ARIMA for time series forecasting.

Authenticate with BigQuery using gcloud auth application-default login if working locally. Create a client object and write a query that selects all columns from sales_history. Use .to_dataframe() to load results into Pandas.

Alternatively, use BigQuery Notebook.

Sort the DataFrame by date and product to ensure chronological order for time series analysis.

Create a Python file called demand_forecaster.ipynb and load the training data

In [ ]:
# Extract all historical data
query = """
SELECT *
FROM `freshfoods.sales_history`
"""

df_history = client.query(query).to_dataframe()

print(f"Loaded {len(df_history)} historical records")
print(df_history.head(10))

Loaded 1620 historical records
     date_day  product_name  discount_percent  is_holiday  sales_qty
0  2024-07-03  Strawberries              0.00           1        345
1  2024-07-18      Avocados              0.07           0        394
2  2024-07-26  Organic Milk              0.00           1        333
3  2024-08-01  Organic Milk              0.12           1        437
4  2024-08-21  Organic Milk              0.00           0        345
5  2024-09-17  Organic Milk              0.11           0        324
6  2024-10-25  Strawberries              0.18           1        354
7  2024-10-29  Strawberries              0.00           1        379
8  2024-11-10      Avocados              0.06           0        484
9  2024-12-03      Avocados              0.00           1        585


Task 3: Build the linear regression model
The first model quantifies how much extra demand results from promotions and holidays. Store managers need to know: if they discount Avocados by 25% during a holiday week, how many additional units should they order?

HINT
Linear regression models the relationship between independent variables (discounts, holidays) and a dependent variable (sales quantity). Create your feature matrix X using discount_percent and is_holiday columns. Create your target vector y using the sales_qty column.

Split your data using sklearn's train_test_split with an 80/20 ratio. Train the model using .fit() on training data. Evaluate performance using .score() on test data. The R-squared score tells you how much variance your model explains.

Extract coefficients using model.coef_ to interpret how each feature impacts sales. A coefficient of 500 on discount_percent means each 0.10 (10%) discount increase drives 50 additional unit sales.

The coefficients reveal business insights. If discount coefficient is 450, then a 20% discount (0.20) generates approximately 90 additional unit sales (450 × 0.20).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Prepare features and target
X = df_history[['discount_percent','is_holiday' ]]
y = df_history['sales_qty']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train linear regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Evaluation
y_pred_lr = lr_model.predict(X_test)
r2_lr = r2_score( y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
#Rappel si tu inverses encore le  y_pred_lr avant le y_test :
# Le linearRegression model fait beaucoup pire qu’un modèle naïf qui prédirait
# simplement la moyenne des ventes chaque jour.

print(f"\n=== Linear Regression Results ===")
print(f"R² Score: {r2_lr:.3f}")
print(f"Mean Absolute Error: {mae_lr:.1f} units")
print(f"Impact coefficients: {lr_model.coef_} ")
print(f"Discount Impact: {lr_model.coef_[0]:.1f} units per 10% discount")
print(f"Holiday Impact: {lr_model.coef_[1]:.1f} units boost")



=== Linear Regression Results ===
R² Score: 0.041
Mean Absolute Error: 85.1 units
Impact coefficients: [10.37275273 42.99515848] 
Discount Impact: 10.4 units per 10% discount
Holiday Impact: 43.0 units boost


MAE : Les prédictions de +- 85 unités/jour

les coef sont significatifs
Une augmentation d'une unité de duscount percent soit 10% =>
une augmentation de 10,4 unités de volumes
Quand is_holiday =1, les ventes augmentent en moyennne de 43 unités,ceteris paribus

Task 4: Build the ARIMA time series model
Linear regression ignores time patterns. Strawberry sales peak in summer. Milk sales spike during winter holidays. The ARIMA model captures these seasonal trends and momentum shifts that regression misses.

HINT
ARIMA requires univariate time series data (single product, date sequence, sales values). Filter your dataset for one product using boolean indexing. Set the date column as the index using .set_index() and ensure it has datetime format.

ARIMA takes three parameters: p (autoregressive order), d (differencing order), q (moving average order). Start with (1,1,1) as a baseline. These parameters control how the model learns from past values, trends, and forecast errors.

Fit the ARIMA model using .fit(). Generate forecasts using .forecast(steps=30) to predict 30 days ahead. The model returns predicted values based purely on historical patterns.

In [12]:

# Filter for single product time series
df_avocado = df_history[df_history['product_name'] == 'Avocados'].copy()

df_avocado['date_day'] = pd.to_datetime(df_avocado['date_day'])
df_avocado = df_avocado.set_index('date_day')

# Train ARIMA model
# Parameters (p,d,q): p=1 autoregressive term, d=1 differencing, q=1 moving average
arima_model = ARIMA(df_avocado['sales_qty'], order=(1, 1, 1))
arima_fitted = arima_model.fit()

print(f"\n=== ARIMA Results ===")
print(arima_fitted.summary())

# Generate 30-day forecast
forecast_30d = arima_fitted.forecast(steps=30)
print(f"\nNext 30 days forecast (mean): {forecast_30d.mean():.1f} units/day")



=== ARIMA Results ===
                               SARIMAX Results                                
Dep. Variable:              sales_qty   No. Observations:                  540
Model:                 ARIMA(1, 1, 1)   Log Likelihood               -3181.479
Date:                Fri, 19 Dec 2025   AIC                           6368.958
Time:                        16:27:51   BIC                           6381.827
Sample:                             0   HQIC                          6373.991
                                - 540                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.0289      0.049      0.591      0.554      -0.067       0.125
ma.L1         -0.9980      0.012    -85.153      0.000      -1.021      -0.975
sigma2      7763.2881    320.

Task 5: Compare model predictions
Store managers need to understand which model answers which question. The regression model tells them how much lift a promotion generates. The ARIMA model tells them what baseline demand looks like without promotions.

HINT
Create a future scenario for testing. Build a DataFrame with 7 days of planned promotions: 20% discount every day, no holidays. Use the linear regression model to predict sales with these promotions using .predict().

Compare these predictions against the ARIMA baseline forecast for the same period. Calculate the incremental lift as the difference between regression prediction and ARIMA baseline. This shows the pure promotional effect.

This comparison reveals the business value. If ARIMA predicts 450 units baseline and regression predicts 520 units with 20% discount, the promotion generates 70 incremental units. Managers multiply this by margin to calculate ROI.

In [13]:
# Create test scenario: 7 days with 20% discount, no holidays
future_scenario = pd.DataFrame({
    'discount_percent': [0.20] * 7,
    'is_holiday': [0] * 7
})

# Regression prediction (with promotion)
lr_forecast = lr_model.predict(future_scenario)

# ARIMA prediction (baseline, no promotion)
arima_forecast = arima_fitted.forecast(steps=7)

# Calculate promotional lift
comparison_df = pd.DataFrame({
    'Day': range(1, 8),
    'Baseline_ARIMA':arima_forecast,
    'With_Promo_LR': lr_forecast,
    'Incremental_Lift': lr_forecast - arima_forecast.values
})

print(f"\n=== Model Comparison ===")
print(comparison_df)
print(f"\nAverage daily lift from 20% discount: {comparison_df['Incremental_Lift'].mean():.1f} units")



=== Model Comparison ===
     Day  Baseline_ARIMA  With_Promo_LR  Incremental_Lift
540    1      500.816528     371.401649       -129.414879
541    2      500.984490     371.401649       -129.582841
542    3      500.989340     371.401649       -129.587691
543    4      500.989480     371.401649       -129.587831
544    5      500.989484     371.401649       -129.587835
545    6      500.989484     371.401649       -129.587835
546    7      500.989484     371.401649       -129.587835

Average daily lift from 20% discount: -129.6 units


In [14]:
print(comparison_df ['Incremental_Lift'].dtype)

float64


Task 6: Generate production forecasts for all products
The VP wants forecasts for all three products covering the next two weeks. You create a production forecast table combining both model types.

HINT
Loop through each product in your dataset using df_history['product_name'].unique(). For each product, filter the data, train an ARIMA model, and generate a 14-day forecast.

Create a date range for the next 14 days using pd.date_range(start=df_history['date_day'].max() + pd.Timedelta(days=1), periods=14). Build a DataFrame containing dates, product names, and ARIMA baseline forecasts.

Use the linear regression model to predict promotional scenarios. Create additional columns showing forecasts with 15% discount and 25% discount applied. This gives managers multiple planning scenarios.

The forecast table now contains baseline projections and promotional scenarios for every product. Store managers review this table Monday morning to plan inventory orders for the week.

In [15]:
df_history.head()

,date_day,product_name,discount_percent,is_holiday,sales_qty
0,2024-07-03,Strawberries,0.00,1,345
1,2024-07-18,Avocados,0.07,0,394
2,2024-07-26,Organic Milk,0.00,1,333
3,2024-08-01,Organic Milk,0.12,1,437
4,2024-08-21,Organic Milk,0.00,0,345


In [ ]:
# Generate future dates for next 14 days
last_date = df_history['date_day'].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=14)

# Store all forecasts
all_forecasts = []

# Loop through products
for product in df_history['product_name'].unique():

    # Get product time series

    # Train ARIMA


    # Create scenarios
    scenario_15pct = pd.DataFrame({
        'discount_percent': ...,
        'is_holiday': ...
    })
    scenario_25pct = pd.DataFrame({
        'discount_percent': ...,
        'is_holiday': ...
    })

    promo_15 = lr_model.predict(scenario_15pct)
    promo_25 = lr_model.predict(scenario_25pct)

    # Build forecast dataframe
    for i, date in enumerate(future_dates):
        all_forecasts.append({
            'forecast_date': ...,
            'product_name': ...,
            'baseline_forecast_arima': int(baseline.iloc[i]),
            'forecast_with_15pct_discount': int(promo_15[i]),
            'forecast_with_25pct_discount': int(promo_25[i])
        })

df_forecasts = pd.DataFrame(all_forecasts)
print(f"\n=== Production Forecasts ===")
print(df_forecasts.head(15))
